In [ ]:
# Databricks notebook sourceMAGIC %md# common feature among policies , members , claims

In [ ]:
# Load tablesclaims_df = spark.table("bupa_bronze.claims")policies_df = spark.table("bupa_bronze.policies")members_df = spark.table("bupa_bronze.members")# Find common columnscommon_cols = set(claims_df.columns) & set(policies_df.columns) & set(members_df.columns)# Intersect data for common columnsclaims_common = claims_df.select(*common_cols)policies_common = policies_df.select(*common_cols)members_common = members_df.select(*common_cols)# Find rows with similar data across all three tables (intersection)common_data = claims_common.intersect(policies_common).intersect(members_common)# Count similar rows in each DataFrameclaims_count = claims_common.intersect(common_data).count()policies_count = policies_common.intersect(common_data).count()members_count = members_common.intersect(common_data).count()result_df = spark.createDataFrame(    [("claims_df", claims_count), ("policies_df", policies_count), ("members_df", members_count)],    ["dataframe", "similar_row_count"])display(result_df)

In [ ]:
# Load tablesclaims_df = spark.table("bupa_bronze.claims")policies_df = spark.table("bupa_bronze.policies")members_df = spark.table("bupa_bronze.members")# Find common columnscommon_cols = set(claims_df.columns) & set(policies_df.columns) & set(members_df.columns)# Intersect data for common columnsclaims_common = claims_df.select(*common_cols)policies_common = policies_df.select(*common_cols)members_common = members_df.select(*common_cols)# Find rows with similar data across all three tables (intersection)common_data = claims_common.intersect(policies_common).intersect(members_common)display(common_data)

%md# claims raw data present in bronze

In [ ]:
MAGIC %sqlselect * from bupa_bronze.claims

%md# Cell 1 — Setup & utils

In [ ]:
MAGIC %run /Workspace/Repos/karthikvanapalli1505@gmail.com/Bupa_Insuarnce_Project/02_Silver_Layer/_00_utils_silver

In [ ]:
# ==============================================# CLAIMS → SILVER  (Providers optional)# ==============================================from pyspark.sql import functions as Ffrom pyspark.sql.types import *DB_SILVER = "bupa_silver"spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_SILVER}")# If you read from paths, keep your dicts; otherwise point to a bronze table# paths_bronze["claims"] must exist if you use read.format("delta").load(...)# Or switch to spark.table("bupa_bronze.claims")

%md# Cell 2 — Read Bronze → Select/Cast (exact columns, tolerant to presence/absence)

In [ ]:
# Pick ONE bronze source style:# clm_bz = spark.read.format("delta").load(paths_bronze["claims"])clm_bz = spark.table("bupa_bronze.claims")   # adjust if different# EXACT mapping (includes Provider_ID + Policy_ID; Member_Key is optional)# If your bronze has Member_Key, we keep it; if not, it won't break anythingcols = clm_bz.columnsdef has(c): return c in colsclm = (clm_bz    .select(        F.col("Claim_ID").cast("string").alias("Claim_ID"),        F.col("Policy_ID").cast("string").alias("Policy_ID"),        *( [F.col("Provider_ID").cast("string").alias("Provider_ID")] if has("Provider_ID") else [] ),        *( [F.col("Member_Key").cast("string").alias("Member_Key")]   if has("Member_Key") else [] ),        F.to_date("Date_Reported").alias("Date_Reported"),        F.to_date("Date_Settled").alias("Date_Settled"),        F.col("Claim_Amount_GBP").cast("double").alias("Claim_Amount_GBP"),        F.col("Payout_GBP").cast("double").alias("Payout_GBP"),        F.col("Fraud_Label").cast("string").alias("Fraud_Label"),        F.initcap(F.trim(F.col("Claim_Type"))).alias("Claim_Type"),        F.initcap(F.trim(F.col("Claim_Status"))).alias("Claim_Status"),    ))print("Bronze → staged rows:", clm.count())clm.printSchema()

%md# Cell 3 — Basic fixes (keys, dates, fraud flag)

In [ ]:
# Key checkclm_bad = clm.filter(F.col("Claim_ID").isNull())clm_ok  = clm.filter(F.col("Claim_ID").isNotNull())if clm_bad.take(1):    quarantine(clm_bad, "null_claim_id", "claims")# Date fixes (swap when settled < reported)clm_ok = fix_dates(clm_ok, "Date_Reported", "Date_Settled")# Fraud flag normalize to '1'/'0' (string) so DQ is stables = F.lower(F.trim(F.col("Fraud_Label")))clm_ok = clm_ok.withColumn(    "Fraud_Flag",    F.when(s.isin("1","y","yes","true","t","on"), "1")     .when(s.isin("0","n","no","false","f","off"), "0")     .otherwise(None))

%md# Cell 4 — Strict FK to Policies using canonicalization + padding (Providers optional)

In [ ]:
# Canonical helper + padding rules from Policiesdef canon(col):    return F.upper(F.regexp_replace(F.trim(F.col(col)), r"[^A-Z0-9_]", ""))# Build claims-side canonical Policy_IDclm_keys = (clm_ok    .withColumn("Policy_ID_raw", F.col("Policy_ID"))    .withColumn("Policy_ID_canon", canon("Policy_ID"))    .select(*[c for c in clm_ok.columns if c not in ["Policy_ID"]], "Policy_ID_raw", "Policy_ID_canon"))# Pull policy canonical keys from Silverpol_keys = (spark.table(f"{DB_SILVER}.policies")    .select(F.col("Policy_ID").cast("string").alias("Policy_ID"))    .withColumn("Policy_ID_canon", canon("Policy_ID"))    .select("Policy_ID_canon").dropDuplicates())# Build per-prefix padding rules from policies (e.g., P_123 → P_000123)prefix_re = r"^([A-Z_]*?)(\d+)$"pol_pad = (pol_keys    .withColumn("prefix", F.regexp_extract("Policy_ID_canon", prefix_re, 1))    .withColumn("digits", F.regexp_extract("Policy_ID_canon", prefix_re, 2))    .withColumn("digit_len", F.length("digits"))    .groupBy("prefix").agg(F.max("digit_len").alias("target_len")))# Apply padding to claims' canonical Policy_IDclm_keys = (clm_keys    .withColumn("m_prefix", F.regexp_extract("Policy_ID_canon", prefix_re, 1))    .withColumn("m_digits", F.regexp_extract("Policy_ID_canon", prefix_re, 2))    .join(pol_pad, F.col("m_prefix") == F.col("prefix"), "left")    .withColumn("Policy_ID_canon",        F.when(F.col("target_len").isNotNull() & (F.col("m_digits") != ""),               F.concat(F.col("m_prefix"), F.lpad(F.col("m_digits"), F.col("target_len"), "0")))         .otherwise(F.col("Policy_ID_canon"))    )    .drop("prefix","digits","digit_len","target_len","m_prefix","m_digits"))# Strict FK enforcement to policies (INNER JOIN on canonical)clm_fk = (clm_keys.join(pol_keys, on="Policy_ID_canon", how="inner")          .drop("Policy_ID_canon"))print("Rows after Policy FK enforcement:", clm_fk.count())# OPTIONAL Providers (soft): if providers table exists & has rows, quarantine bad Provider_IDs but KEEP rowsproviders_exists = Falsetry:    providers_exists = spark._jsparkSession.catalog().tableExists(DB_SILVER, "providers") and \                       spark.table(f"{DB_SILVER}.providers").limit(1).count() > 0except:    providers_exists = Falseif providers_exists and "Provider_ID" in clm_fk.columns:    prov_keys = spark.table(f"{DB_SILVER}.providers").select("Provider_ID").dropDuplicates()    joined = (clm_fk.alias("c")              .join(prov_keys.withColumnRenamed("Provider_ID","_Provider_ID"), F.col("c.Provider_ID") == F.col("_Provider_ID"), "left"))    bad_prov = joined.filter(F.col("c.Provider_ID").isNotNull() & F.col("_Provider_ID").isNull()) \                     .select([F.col("c."+x) for x in clm_fk.columns])    good_prov = joined.filter(F.col("_Provider_ID").isNotNull() | F.col("c.Provider_ID").isNull()) \                      .select([F.col("c."+x) for x in clm_fk.columns])    if bad_prov.take(1):        quarantine(bad_prov, "fk_provider_missing_soft", "claims")    clm_fk = good_provprint("Rows after Provider soft-check (or ignored):", clm_fk.count())

%md# Cell 5 — Deduplicate & feature hints

In [ ]:
# Deduplicate by Claim_ID (keep latest settled, then reported)claims_dedup = drop_dupes_keep_latest(clm_fk, ["Claim_ID"], ["Date_Settled","Date_Reported"])# Featuresclaims_enr = (claims_dedup    .withColumn("Payout_to_Claim_Ratio",        F.when(F.col("Claim_Amount_GBP") > 0, F.round(F.col("Payout_GBP")/F.col("Claim_Amount_GBP"), 4)).otherwise(None))    .withColumn("Claim_Duration_Days",        F.when(F.col("Date_Settled").isNotNull() & F.col("Date_Reported").isNotNull(),               F.datediff("Date_Settled", "Date_Reported")).otherwise(None))    .withColumn("Year_Reported", F.year("Date_Reported")))print("claims_enr rows:", claims_enr.count())

%md# Cell 6 — DQ (critical rules only)

In [ ]:
dq_expect(claims_enr, "key_not_null",          "Claim_ID IS NOT NULL AND Policy_ID_raw IS NOT NULL",   # strict only on Policy (Provider optional)          "error","claims",quarantine,"null_key_claim_or_policy")dq_expect(claims_enr, "date_order",          "Date_Reported IS NULL OR Date_Settled IS NULL OR Date_Settled >= Date_Reported",          "error","claims",quarantine,"date_order_violation")dq_expect(claims_enr, "money_non_negative",          "coalesce(Claim_Amount_GBP,0) >= 0 AND coalesce(Payout_GBP,0) >= 0",          "error","claims",quarantine,"negative_money_values")dq_expect(claims_enr, "payout_le_claim",          "Payout_GBP IS NULL OR Claim_Amount_GBP IS NULL OR Payout_GBP <= Claim_Amount_GBP",          "error","claims",quarantine,"payout_exceeds_claim")

%md# Cell 7 — Write Silver (Delta), external table, OPTIMIZE

In [ ]:
# ---------- Cell 7 — Write Silver (DB + External ADLS) ----------from pyspark.sql import functions as Ffull_table  = f"{DB_SILVER}.claims"PATH_CLAIMS = "abfss://silver@bupaprocesseddatastorage.dfs.core.windows.net/claims"# --- 1) Write to managed Delta table in metastore (bupa_silver.claims)(claims_enr.write    .format("delta")    .mode("overwrite")    .option("overwriteSchema", "true")    .saveAsTable(full_table))print(f"✅ Managed table written: {full_table}")# --- 2) Also save to external ADLS container (same data files)(claims_enr.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(paths_silver["claims"]))print(f"✅ External Delta written to ADLS path:\n   {PATH_CLAIMS}")"""# --- 3) (Optional) Register external table to point at ADLS path# Drop old managed table and recreate it as an external reference, if you want unified locationspark.sql(f"DROP TABLE IF EXISTS {full_table}")spark.sql(f"CREATE TABLE {full_table}USING DELTALOCATION '{PATH_CLAIMS}'")print(f"🔗 Table re-registered at ADLS location: {PATH_CLAIMS}") """# --- 4) Optimize / Z-Order (best effort)try:    spark.sql(f"OPTIMIZE {full_table} ZORDER BY (Policy_ID_raw, Date_Reported)")    print("✅ OPTIMIZE complete.")except Exception as e:    print("[WARN] OPTIMIZE skipped or unavailable:", e)

In [ ]:
MAGIC %sqlselect * from bupa_silver.claimsLIMIT 10

%md# Cell 9 — Metrics

In [ ]:
from datetime import datetimespark.sql(f"""CREATE TABLE IF NOT EXISTS {DB_SILVER}.__run_metrics (  metric STRING, value STRING, context STRING, ts TIMESTAMP) USING DELTA""")def write_metric(n,v,c):    spark.createDataFrame([(n,str(v),c,datetime.utcnow())],        "metric STRING, value STRING, context STRING, ts TIMESTAMP") \      .write.mode("append").saveAsTable(f"{DB_SILVER}.__run_metrics")tbl = spark.table(full_table)write_metric("rowcount_claims_silver", tbl.count(), "claims_silver")write_metric("distinct_claim_ids",     tbl.select("Claim_ID").distinct().count(), "claims_silver")avg_ratio = tbl.select(F.avg("Payout_to_Claim_Ratio")).first()[0]write_metric("claims_paid_ratio_avg",  0 if avg_ratio is None else avg_ratio, "claims_silver")print("📈 Metrics written.")

%md# Cell 10 — Exit

In [ ]:
try:    dbutils.notebook.exit("SUCCESS: claims_silver completed")except Exception as e:    print("SUCCESS: claims_silver completed")

%md# 3) Resolve Providers in Claims (soft now → strict later)

In [ ]:
def canon(c): return F.upper(F.regexp_replace(F.trim(F.col(c)), r"[^A-Z0-9_]", ""))claims = spark.table("bupa_silver.claims").withColumn("Provider_ID_canon", canon("Provider_ID"))prov   = (spark.table("bupa_silver.providers")          .withColumn("Provider_ID_canon", canon("Provider_ID"))          .select("Provider_ID","Provider_ID_canon").dropDuplicates())mp = (spark.table("bupa_reference.map_provider_id")      .withColumn("source_canon", canon("source_provider_id"))      .withColumn("target_canon", canon("target_provider_id"))      .select("source_canon","target_canon"))resolved = (claims.alias("c")  .join(mp.alias("m"), F.col("c.Provider_ID_canon")==F.col("m.source_canon"), "left")  .join(prov.alias("p"),        F.coalesce(F.col("m.target_canon"), F.col("c.Provider_ID_canon"))==F.col("p.Provider_ID_canon"),        "left"))# quarantine unresolved but KEEP the rows (soft enforcement)unresolved = resolved.filter(F.col("p.Provider_ID_canon").isNull()) \                     .select([F.col("c."+x) for x in claims.columns])if unresolved.take(1):    quarantine(unresolved, "fk_provider_unresolved_soft", "claims")# keep the original schema for downstreamclaims_resolved = resolved.select([F.col("c."+x) for x in claims.columns])print("Resolved provider rate:",      round(100.0 * (resolved.filter(F.col("p.Provider_ID_canon").isNotNull()).count() / claims.count()), 2), "%")

%md# 4) Track progress (metrics)

In [ ]:
total = claims.select("Provider_ID").filter("Provider_ID IS NOT NULL").count()mapped = resolved.filter(F.col("p.Provider_ID_canon").isNotNull()).count()rate = 0.0 if total==0 else round(100.0*mapped/total,2)print({"claims_provider_mapping_rate_pct": rate})

In [ ]:
MAGIC %sqlselect * from bupa_reference.map_provider_id

%md# Step 1 — Confirm what’s still unmapped (quick diagnostics)

In [ ]:
from pyspark.sql import functions as Fdef canon(c):     return F.upper(F.regexp_replace(F.trim(F.col(c)), r"[^A-Z0-9_]", ""))claims = spark.table("bupa_silver.claims").withColumn("Provider_ID_canon", canon("Provider_ID"))prov   = spark.table("bupa_silver.providers").withColumn("Provider_ID_canon", canon("Provider_ID")) \                    .select("Provider_ID_canon").dropDuplicates()mp = spark.table("bupa_reference.map_provider_id") \          .withColumn("source_canon", canon("source_provider_id")) \          .withColumn("target_canon", canon("target_provider_id")) \          .select("source_canon","target_canon")# distinct claim provider idsclaim_ids = claims.filter(F.col("Provider_ID_canon").isNotNull()) \                  .select("Provider_ID_canon").distinct()# unmapped sources = claim ids missing in mappingunmapped_sources = claim_ids.join(mp.select("source_canon").distinct(),                                  claim_ids["Provider_ID_canon"]==F.col("source_canon"),                                  "left_anti") \                            .select(F.col("Provider_ID_canon").alias("source_provider_id"))print("distinct claim provider ids:", claim_ids.count())print("distinct providers master ids:", prov.count())print("unmapped claim provider ids:", unmapped_sources.count())display(unmapped_sources.limit(25))

%md--------------------------------

In [ ]:
MAGIC %sqlselect * from bupa_bronze.claims

%md# Differences b/w bronze claims and silver claims

In [ ]:
# Load DataFramesbronze_df = spark.table("bupa_bronze.claims")silver_df = spark.table("bupa_silver.claims")bronze_cols = set(bronze_df.columns)silver_cols = set(silver_df.columns)# Identify new, dropped, and common featuresnew_features = silver_cols - bronze_colsdropped_features = bronze_cols - silver_colscommon_features = sorted(bronze_cols & silver_cols)print(f"New features added in silver: {sorted(new_features)}")print(f"Features dropped in silver: {sorted(dropped_features)}")# Get data types as dictsbronze_types = dict(bronze_df.dtypes)silver_types = dict(silver_df.dtypes)# Align on Claim_ID for accurate comparisonbronze_aligned = bronze_df.select("Claim_ID", *common_features).dropDuplicates(["Claim_ID"])silver_aligned = silver_df.select("Claim_ID", *common_features).dropDuplicates(["Claim_ID"])joined = (    bronze_aligned.alias("bz")    .join(silver_aligned.alias("sv"), on="Claim_ID", how="outer"))# For each common feature, check for changes and count them, and compare typesfor col in common_features:    bronze_type = bronze_types.get(col)    silver_type = silver_types.get(col)    type_changed = bronze_type != silver_type    type_msg = (        f"Type changed: {bronze_type} → {silver_type}"        if type_changed else        f"Type unchanged: {bronze_type}"    )    n_diff = (        joined        .filter(            (joined[f"bz.{col}"].isNull() & joined[f"sv.{col}"].isNotNull()) |            (joined[f"bz.{col}"].isNotNull() & joined[f"sv.{col}"].isNull()) |            (joined[f"bz.{col}"] != joined[f"sv.{col}"])        )        .count()    )    if n_diff == 0 and not type_changed:        print(f"Feature '{col}' is unchanged between bronze and silver. {type_msg}")    else:        print(f"Feature '{col}' was transformed between bronze and silver in {n_diff} claims. {type_msg}")        # Show a sample of differences        sample_diff = (            joined            .filter(                (joined[f"bz.{col}"].isNull() & joined[f"sv.{col}"].isNotNull()) |                (joined[f"bz.{col}"].isNotNull() & joined[f"sv.{col}"].isNull()) |                (joined[f"bz.{col}"] != joined[f"sv.{col}"])            )            .select("Claim_ID", f"bz.{col}", f"sv.{col}")            .limit(5)        )        print(f"Sample differences for '{col}':")        display(sample_diff)# Explain new features and show sample datafor col in sorted(new_features):    print(f"Feature '{col}' added: Reason unknown, please check transformation logic.")    print(f"Sample data for new feature '{col}':")    display(silver_df.select("Claim_ID", col).limit(5))# Note dropped featuresfor col in sorted(dropped_features):    print(f"Feature '{col}' was present in bronze but dropped in silver.")

%md# A) Snapshot basics: row counts & distinct IDs

In [ ]:
from pyspark.sql import functions as FBRONZE_TBL = "bupa_bronze.claims"   # change if your bronze table name differsSILVER_TBL = "bupa_silver.claims"br = spark.table(BRONZE_TBL)sv = spark.table(SILVER_TBL)metrics = {    "rows_bronze": br.count(),    "rows_silver": sv.count(),    "distinct_claim_id_bronze": br.select("Claim_ID").distinct().count(),    "distinct_claim_id_silver": sv.select("Claim_ID").distinct().count(),}print(metrics)

%md# B) Data quality improvement deltas

In [ ]:
from pyspark.sql import functions as Fdef pct(x, n):     return 0.0 if n == 0 else round(100.0 * x / n, 4)# Bronze issuesn_br = br.count()br_null_claim = br.filter(F.col("Claim_ID").isNull()).count()br_bad_dates  = br.filter(F.col("Date_Reported").isNotNull() & F.col("Date_Settled").isNotNull() & (F.col("Date_Settled") < F.col("Date_Reported"))).count()br_neg_money  = br.filter((F.col("Claim_Amount_GBP") < 0) | (F.col("Payout_GBP") < 0)).count()br_payout_gt  = br.filter(F.col("Claim_Amount_GBP").isNotNull() & F.col("Payout_GBP").isNotNull() & (F.col("Payout_GBP") > F.col("Claim_Amount_GBP"))).count()# Silver issues (should be ~0 after DQ gates)n_sv = sv.count()sv_null_claim = sv.filter(F.col("Claim_ID").isNull()).count()sv_bad_dates  = sv.filter(F.col("Date_Reported").isNotNull() & F.col("Date_Settled").isNotNull() & (F.col("Date_Settled") < F.col("Date_Reported"))).count()sv_neg_money  = sv.filter((F.col("Claim_Amount_GBP") < 0) | (F.col("Payout_GBP") < 0)).count()sv_payout_gt  = sv.filter(F.col("Claim_Amount_GBP").isNotNull() & F.col("Payout_GBP").isNotNull() & (F.col("Payout_GBP") > F.col("Claim_Amount_GBP"))).count()report = {    "bronze_bad_claim_id_pct": pct(br_null_claim, n_br),    "silver_bad_claim_id_pct": pct(sv_null_claim, n_sv),    "bronze_bad_date_order_pct": pct(br_bad_dates, n_br),    "silver_bad_date_order_pct": pct(sv_bad_dates, n_sv),    "bronze_negative_money_pct": pct(br_neg_money, n_br),    "silver_negative_money_pct": pct(sv_neg_money, n_sv),    "bronze_payout_gt_claim_pct": pct(br_payout_gt, n_br),    "silver_payout_gt_claim_pct": pct(sv_payout_gt, n_sv),}report

%md# C) FK progress vs Policies (strict in Silver)

In [ ]:
def canon(col):    return F.upper(F.regexp_replace(F.trim(F.col(col)), r"[^A-Z0-9_]", ""))pol = spark.table("bupa_silver.policies").select(canon("Policy_ID").alias("Policy_ID_canon")).dropDuplicates()# Bronze FK miss (no padding repair here → "as-is")br_fk_miss = (br  .withColumn("Policy_ID_canon", canon("Policy_ID"))  .join(pol, "Policy_ID_canon", "left_anti")).count()br_fk_pct = pct(br_fk_miss, br.count())# Silver should be ~0 because we enforce inner join to policiessv_fk_miss = (sv  .withColumn("Policy_ID_canon", canon("Policy_ID_raw"))   # we kept Policy_ID_raw from claims  .join(pol, "Policy_ID_canon", "left_anti")).count()sv_fk_pct = pct(sv_fk_miss, sv.count())print({"bronze_fk_missing_pct": br_fk_pct, "silver_fk_missing_pct": sv_fk_pct})

%md# D) Feature coverage check (Silver-only)

In [ ]:
exists = {c: (c in sv.columns) for c in ["Payout_to_Claim_Ratio","Claim_Duration_Days","Year_Reported","Fraud_Flag"]}exists